# Scheduled DMT Evolution in Operator Space

This notebook shows how the operator-space DMT API uses the same schedule idea as TEBD.

The split is:

- the **schedule** says where gates act and in what order
- `dmt_step!` is the low-level single-update kernel
- `DMTGateEvolution(...)` packages a scheduled DMT sweep
- `dmt_evolve!` (or `evolve!`) runs the scheduled evolution

We will use a simple Pauli-basis operator as the initial state and evolve it with a nearest-neighbor operator-space gate.


In [ ]:
using LinearAlgebra
using Plots
using ITensors
using ITensorMPS
using MPSToolkit

default(size=(960, 680), linewidth=2, markersize=4, legend=:best)

function local_pauli_z_profile(state::MPS)
    sites = siteinds(state)
    nsites = length(state)
    return [
        real(inner(pauli_basis_state(sites, [site == j ? "Z" : "I" for site in 1:nsites]), state))
        for j in 1:nsites
    ]
end

spins = spinhalf_matrices()
function operator_space_gate(dt; Jxy::Real=1.0, Delta::Real=0.8)
    h = Jxy * kron(spins.Sx, spins.Sx) +
        Jxy * kron(spins.Sy, spins.Sy) +
        Delta * kron(spins.Sz, spins.Sz)
    return pauli_gate(exp(-im * dt * h))
end


## Pauli-basis setup

`pauli_siteinds(...)` gives a local dimension-4 site set representing the basis `(I, X, Y, Z)`.
We start from a simple operator with alternating `Z` and identity support.


In [ ]:
nsites = 6
sites = pauli_siteinds(nsites)
state = pauli_basis_state(sites, [isodd(n) ? "Z" : "I" for n in 1:nsites])
normalize!(state)

bar(1:nsites, local_pauli_z_profile(state); xlabel="site", ylabel="Pauli-basis amplitude", title="Initial local Z profile", label="t = 0")


## Building a scheduled DMT sweep

A standard nearest-neighbor DMT sweep uses:

- a forward schedule, usually `1:(nsites - 1)`
- a reverse schedule, usually `(nsites - 1):-1:1`
- one local operator-space gate per scheduled bond

This is the same schedule concept as TEBD. The difference is that each scheduled update is applied with `dmt_step!` under the hood.


In [ ]:
dt = 0.05
gate = operator_space_gate(dt)
forward_schedule = collect(1:(nsites - 1))
reverse_schedule = collect((nsites - 1):-1:1)
gates = fill(gate, length(forward_schedule))

evo = DMTGateEvolution(
    gates,
    dt;
    schedule=forward_schedule,
    reverse_schedule=reverse_schedule,
    nstep=1,
    maxdim=16,
    cutoff=1e-10,
    gate_maxdim=128,
)

state_scheduled = copy(state)
profiles = [local_pauli_z_profile(state_scheduled)]
for _ in 1:6
    dmt_evolve!(state_scheduled, evo)
    push!(profiles, local_pauli_z_profile(state_scheduled))
end

heatmap(
    0:6,
    1:nsites,
    reduce(hcat, profiles);
    xlabel="DMT sweep step",
    ylabel="site",
    title="Scheduled DMT evolution in operator space",
    colorbar_title="local Z amplitude",
)


## The low-level view: `dmt_step!`

When you want full control, you can call `dmt_step!` directly. That is useful for experiments, custom adaptive schedules, or debugging a specific update.
The scheduled driver is mainly a convenience layer around repeated low-level steps.


In [ ]:
state_manual = copy(state)
for bond in forward_schedule
    dmt_step!(state_manual, gate, bond; maxdim=16, cutoff=1e-10, direction=:R, gate_maxdim=128)
end
for bond in reverse_schedule
    dmt_step!(state_manual, gate, bond; maxdim=16, cutoff=1e-10, direction=:L, gate_maxdim=128)
end
normalize!(state_manual)

state_one_sweep = copy(state)
dmt_evolve!(state_one_sweep, evo)

println("overlap(manual, scheduled after one sweep) = ", abs(inner(state_manual, state_one_sweep)))
plot(
    1:nsites,
    local_pauli_z_profile(state_manual);
    label="manual dmt_step! sweep",
    xlabel="site",
    ylabel="local Z amplitude",
    title="Manual vs scheduled DMT uses the same kernel",
)
plot!(1:nsites, local_pauli_z_profile(state_one_sweep); label="scheduled dmt_evolve!")


## Closing Notes

The scheduler story is now the same in both workflows:

- TEBD: schedule + local gate + `LocalGateEvolution`
- DMT: schedule + local gate + `DMTGateEvolution`

The difference is not the scheduling idea. The difference is the single-step backend that gets applied at each scheduled update.
